# Thí nghiệm thăm dò: độ phân tán theo seed trong MMKGC

**Câu hỏi:** độ lệch chuẩn của MRR qua các seed có lớn hơn khoảng cách mà các bài MMKGC dùng để tuyên bố vượt SOTA hay không?

Mô hình khảo sát: **AdaMF-MAT** (LREC-COLING 2024) — repo hoàn chỉnh nhất trong nhóm, có script cho cả ba bộ dữ liệu và có tham số seed hoạt động.

**Trước khi chạy:** Settings → Accelerator → **GPU T4 x2** (hoặc P100), và bật Internet.

---

### Nguyên tắc của thí nghiệm này

- Siêu tham số **đọc trực tiếp từ script của tác giả**, không tự đặt
- Chỉ thay đổi duy nhất `-seed`
- Mọi con số đối chiếu đều trích từ bảng trong bài báo gốc, có ghi nguồn
- Kết quả nói về **những gì được công bố**, không suy ra tác giả đã làm gì

## 1. Môi trường

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Tải mã nguồn

In [ ]:
%cd /kaggle/working
!rm -rf AdaMF-MAT
!git clone -q https://github.com/zjukg/AdaMF-MAT.git
!ls AdaMF-MAT

## 3. Biên dịch thư viện C++

`mmkgc/` là bản sửa đổi của OpenKE. Phần lấy mẫu âm và tính chỉ số đánh giá nằm trong C++, phải biên dịch cho đúng kiến trúc máy.

In [ ]:
%cd /kaggle/working/AdaMF-MAT/mmkgc
!bash make.sh
!ls -la release/
%cd /kaggle/working

## 4. Vá tương thích thiết bị

Repo gọi `.cuda()` cứng ở 28 vị trí. Ô dưới thay bằng một module chọn thiết bị, để mã chạy được cả khi không có GPU NVIDIA.

In [ ]:
import os, glob, re

A = "/kaggle/working/AdaMF-MAT"

# --- module chon thiet bi ---
os.makedirs(f"{A}/mmkgc", exist_ok=True)
with open(f"{A}/mmkgc/device.py", "w") as f:
    f.write("""import os
import torch
_f = os.environ.get("MMKGC_DEVICE", "").strip().lower()
if _f:
    DEVICE = torch.device(_f)
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
def device_name():
    return str(DEVICE)
""")

total = 0
for p in glob.glob(f"{A}/mmkgc/**/*.py", recursive=True):
    if p.endswith("device.py"):
        continue
    s = open(p, encoding="utf-8").read()
    if ".cuda()" not in s:
        continue
    n = s.count(".cuda()")
    depth = os.path.relpath(p, f"{A}/mmkgc").count(os.sep)
    imp = "from " + "." * (depth + 1) + "device import DEVICE"
    if "device import DEVICE" not in s:
        lines = s.split("\n")
        last = max((i for i, l in enumerate(lines) if l.startswith(("import ", "from "))), default=0)
        lines.insert(last + 1, imp)
        s = "\n".join(lines)
    s = s.replace(".cuda()", ".to(DEVICE)")
    open(p, "w", encoding="utf-8").write(s)
    total += n
    print(f"  {os.path.relpath(p, A):<42} {n}")

p = f"{A}/run_adamf_mat.py"
s = open(p, encoding="utf-8").read()
if "from mmkgc.device import DEVICE" not in s:
    s = s.replace("from args import get_args",
                  "from args import get_args\nfrom mmkgc.device import DEVICE, device_name")
s = s.replace("use_gpu=True", "use_gpu=(DEVICE.type != 'cpu')")
open(p, "w", encoding="utf-8").write(s)
print(f"\nDa thay {total} cho .cuda()")


## 5. Tải bộ nhúng

Đặc trưng thị giác và văn bản đã trích sẵn, không nằm trong repo.

Nguồn: https://drive.google.com/drive/folders/1UJSfnb8DEx2s-k8zaQx1fWUw5f45GBpI

In [ ]:
!pip install -q gdown
%cd /kaggle/working/AdaMF-MAT
!gdown --folder 1UJSfnb8DEx2s-k8zaQx1fWUw5f45GBpI -O ./emb_dl -q

import shutil, pathlib
pathlib.Path('embeddings').mkdir(exist_ok=True)
for f in pathlib.Path('emb_dl').rglob('*.pth'):
    shutil.move(str(f), f'embeddings/{f.name}')
!ls -la embeddings/

### Kiểm chứng số chiều đặc trưng

Bước này cũng là một quan sát cho nghiên cứu: các bộ dữ liệu dùng encoder khác nhau nên số chiều chênh nhau hơn mười lần.

In [ ]:
import torch, os
print(f"{'file':<24}{'shape':>18}")
print('-'*42)
for f in sorted(os.listdir('embeddings')):
    if f.endswith('.pth'):
        t = torch.load(f'embeddings/{f}', map_location='cpu', weights_only=False)
        print(f'{f:<24}{str(tuple(t.shape)):>18}')

## 6. Bộ khung chạy thí nghiệm

In [ ]:
import subprocess, sys, os, re, json, time, statistics

A = "/kaggle/working/AdaMF-MAT"
OUT = "/kaggle/working/results"
os.makedirs(OUT, exist_ok=True)

METRIC_LINE = re.compile(
    r"^\s*(l|r|averaged)\(filter\):\s*"
    r"([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)",
    re.MULTILINE)

SHELL_VAR = re.compile(r"^\s*([A-Z_][A-Z0-9_]*)=(\S+)\s*$", re.MULTILINE)
SCRIPT_FOR = {"MKG-W": "run_mkgw.sh", "MKG-Y": "run_mkgy.sh", "DB15K": "run_db15k.sh"}


def read_author_config(dataset):
    """Doc sieu tham so tu chinh script cua tac gia, khong tu dat."""
    path = f"{A}/scripts/{SCRIPT_FOR[dataset]}"
    cfg = dict(SHELL_VAR.findall(open(path).read()))
    assert cfg.get("DATA") == dataset, f"{path} khai bao DATA={cfg.get('DATA')}"
    for k in ["EMB_DIM", "NUM_BATCH", "MARGIN", "LR", "NEG_NUM", "EPOCH"]:
        assert k in cfg, f"{path} thieu {k}"
    return cfg


def parse_metrics(text):
    sides = {}
    for m in METRIC_LINE.finditer(text):
        sides[m.group(1)] = dict(
            mrr=float(m.group(2)), mr=float(m.group(3)),
            hit10=float(m.group(4)), hit3=float(m.group(5)),
            hit1=float(m.group(6)))
    if "averaged" in sides:
        return sides["averaged"]
    if "l" in sides and "r" in sides:
        return {k: (sides["l"][k] + sides["r"][k]) / 2
                for k in ("mrr", "mr", "hit10", "hit3", "hit1")}
    return None


def run_seed(dataset, seed, cfg, epoch=None):
    n_epoch = epoch or int(cfg["EPOCH"])
    save = f"{A}/checkpoint/seed{seed}"
    os.makedirs(f"{A}/checkpoint", exist_ok=True)
    cmd = [sys.executable, "run_adamf_mat.py",
           f"-dataset={dataset}", f"-batch_size={cfg['NUM_BATCH']}",
           f"-margin={cfg['MARGIN']}", f"-epoch={n_epoch}",
           f"-dim={cfg['EMB_DIM']}", f"-neg_num={cfg['NEG_NUM']}",
           f"-learning_rate={cfg['LR']}", "-mu=0",
           f"-seed={seed}", f"-save={save}"]

    print(f"  seed {seed} ...", flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, cwd=A, capture_output=True, text=True)
    dt = time.time() - t0

    log = f"{OUT}/{dataset}_seed{seed}.log"
    with open(log, "w") as f:
        f.write(r.stdout + "\n===STDERR===\n" + r.stderr)

    m = parse_metrics(r.stdout + r.stderr)
    if m is None:
        print(f"    that bai sau {dt:.0f}s — xem {log}")
        print("    " + "\n    ".join(r.stdout.strip().splitlines()[-8:]))
        return None
    m["seed"] = seed
    m["seconds"] = round(dt, 1)
    print(f"    MRR={m['mrr']:.4f}  H@1={m['hit1']:.4f}  ({dt/60:.1f} phut)")
    return m


def summarize(runs, dataset, cfg, n_epoch):
    if len(runs) < 2:
        print("Khong du ket qua de tinh do phan tan.")
        return None

    print("\n" + "=" * 68)
    print(f"KET QUA — AdaMF-MAT tren {dataset}, {len(runs)} seed")
    print("=" * 68)
    print(f"{'seed':>6} {'MRR':>9} {'H@1':>9} {'H@3':>9} {'H@10':>9}")
    for r in runs:
        print(f"{r['seed']:>6} {r['mrr']:>9.4f} {r['hit1']:>9.4f} "
              f"{r['hit3']:>9.4f} {r['hit10']:>9.4f}")

    print()
    stats = {}
    for k in ("mrr", "hit1", "hit3", "hit10"):
        v = [r[k] for r in runs]
        stats[k] = dict(mean=statistics.mean(v), std=statistics.stdev(v),
                        lo=min(v), hi=max(v), spread=max(v) - min(v))
        s = stats[k]
        print(f"{k.upper():>6}: {s['mean']:.4f} +/- {s['std']:.4f}"
              f"   [{s['lo']:.4f}, {s['hi']:.4f}]   bien do {s['spread']:.4f}")

    payload = dict(model="AdaMF-MAT", dataset=dataset, epoch=n_epoch,
                   config_source=f"scripts/{SCRIPT_FOR[dataset]}",
                   config=cfg, runs=runs, stats=stats)
    with open(f"{OUT}/{dataset}_summary.json", "w") as f:
        json.dump(payload, f, indent=2)
    return stats


In [ ]:
# Doi chieu voi khoang cach da cong bo.
# MOI con so duoi day duoc doc truc tiep tu bang trong bai bao goc.

PUBLISHED_GAPS = [
    dict(dataset="DB15K", label="APKGC vs NativE", gap=0.002,
         source="APKGC AAAI 2025, Bang 2, tr.15010 (0.364 vs 0.362)"),
    dict(dataset="MKG-W", label="APKGC vs NativE", gap=0.008,
         source="APKGC AAAI 2025, Bang 2, tr.15010 (0.374 vs 0.366)"),
    dict(dataset="MKG-W", label="APKGC: dong gop attention penalty", gap=0.014,
         source="APKGC AAAI 2025, Bang 4, tr.15011 (0.374 vs 0.360)"),
    dict(dataset="MKG-W", label="APKGC: dong gop noise sampling", gap=0.016,
         source="APKGC AAAI 2025, Bang 4, tr.15011 (0.374 vs 0.358)"),
]

def interpret(stats, dataset):
    std = stats["mrr"]["std"]
    print("=" * 68)
    print("DIEN GIAI")
    print("=" * 68)
    print(f"Do lech chuan MRR do duoc tren {dataset}: {std:.4f}\n")
    print("Doi chieu voi khoang cach giua cac phuong phap trong bang da cong bo:")
    for g in PUBLISHED_GAPS:
        if g["dataset"] != dataset:
            continue
        v = "NAM TRONG nhieu seed" if std >= g["gap"] else "vuot nhieu seed"
        print(f"  {g['label']:<38} {g['gap']:.4f}  ->  {v}")
        print(f"    nguon: {g['source']}")

    print()
    if std > 0.005:
        print("KET LUAN: do phan tan giua cac seed lon hon phan lon khoang cach")
        print("duoc cong bo. Luan diem cua nghien cuu dung vung.")
    elif std < 0.002:
        print("KET LUAN: mo hinh on dinh giua cac seed. Cac cai tien cong bo")
        print("nhieu kha nang la that. Nen chuyen trong tam sang phan audit")
        print("tham so va tinh khong dong nhat cua encoder.")
    else:
        print("KET LUAN: chua dut khoat. Can them seed hoac them mo hinh.")

    print()
    print("LUU Y KHI DIEN GIAI:")
    print("Ket qua nay cho biet do phan tan CUA LAN CHAY NAY voi cau hinh cong")
    print("bo. No KHONG cho phep ket luan tac gia da chay bao nhieu lan hay da")
    print("thu bao nhieu cau hinh — repo chi phan anh nhung gi duoc cong bo.")


## 7. Chạy thử một seed

Kiểm tra pipeline thông suốt trước khi chạy toàn bộ. Đặt số epoch nhỏ.

In [ ]:
cfg = read_author_config('MKG-W')
print('Cau hinh doc tu script tac gia:', cfg)
print()
smoke = run_seed('MKG-W', 42, cfg, epoch=20)

## 8. Chạy đầy đủ — 5 seed

Dùng đúng số epoch trong script của tác giả. Thời gian phụ thuộc GPU; trên T4 dự kiến vài giờ cho cả năm lần chạy.

In [ ]:
DATASET = 'MKG-W'
SEEDS = [42, 1234, 2024, 3407, 7]

cfg = read_author_config(DATASET)
n_epoch = int(cfg['EPOCH'])
print(f'Mo hinh   : AdaMF-MAT')
print(f'Du lieu   : {DATASET}')
print(f'Cau hinh  : {cfg}')
print(f'Cac seed  : {SEEDS}')
print()

runs = []
for sd in SEEDS:
    r = run_seed(DATASET, sd, cfg)
    if r:
        runs.append(r)

stats = summarize(runs, DATASET, cfg, n_epoch)

## 9. Diễn giải

In [ ]:
if stats:
    interpret(stats, DATASET)

## 10. Lưu kết quả

In [ ]:
!ls -la /kaggle/working/results/
import json, glob
for f in glob.glob('/kaggle/working/results/*_summary.json'):
    print('\n' + f)
    print(json.dumps(json.load(open(f))['stats'], indent=2))

---

### Ghi chú về phạm vi kết luận

Thí nghiệm này đo độ phân tán của **một mô hình** trên **một bộ dữ liệu** với cấu hình được công bố. Nó cho biết mức nhiễu do seed gây ra trong điều kiện đó.

Nó **không** cho phép kết luận tác giả đã chạy bao nhiêu lần hay thử bao nhiêu cấu hình — kho mã chỉ phản ánh những gì được công bố, không phản ánh toàn bộ quá trình nghiên cứu. Phát biểu kiểm chứng được là *bài báo không báo cáo độ lệch chuẩn*, không phải *tác giả không đo độ lệch chuẩn*.